In [ ]:
'''READ ME: Just hit control + Enter to run! (on windows). The interactive display will be shown at the end of the cell'''

# ---- import statements; we need to import specific packages that our environment can then use ----
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from ipywidgets import interact, Dropdown, IntSlider, fixed, Button, Output, VBox
from IPython.display import Audio, display

from scipy.fft import irfft, rfftfreq

# ---- dataset -----
# use digitized data from Hillenbrand 1995, which gives the sound duration, f0, and measured formant (F1-F4) frequencies for different vowels
vowel_labels = ['heed, /i/', 'hid, /I/', 'hayed, /e/', 'head, /ε/', 'had, /æ/', 'hod, /α/', 'hawed, /ɔ/', 'hoed, /o/', 'hood, /U/', 'who\'d, /u/', 'hud, /Λ/', 'heard, /ɝ/'] 

vowels   = ['/i/', '/I/', '/e/', '/ε/', '/æ/', '/α/', '/ɔ/', '/o/', '/U/', '/u/', '/Λ/', '/ɝ/']  # phoneme labels for each vowel
groups   = ['Man', 'Woman', 'Child'] # men, women, children (different groups surveyed in Hillenbrand)
measures = ['Dur', 'F0', 'F1', 'F2', 'F3', 'F4']

# define data table from Hillenbrand data set
data = np.array([
    # Dur (ms)
    [[243,192,267,189,278,267,283,265,192,237,188,263],
     [306,237,320,254,332,323,353,326,249,303,226,321],
     [297,248,314,235,322,311,319,310,247,278,234,307]],
    # F0 (Hz)
    [[138,135,129,127,123,123,121,129,133,143,133,130],
     [227,224,219,214,215,215,210,217,230,235,218,217],
     [246,241,237,230,228,229,225,236,243,249,236,237]],
    # F1 (Hz)
    [[342,427,476,580,588,768,652,497,469,378,623,474],
     [437,483,536,731,669,936,781,555,519,459,753,523],
     [452,511,564,749,717,1002,803,597,568,494,749,586]],
    # F2 (Hz)
    [[2322,2034,2089,1799,1952,1333,997,910,1122,997,1200,1379],
     [2761,2365,2530,2058,2349,1551,1136,1035,1225,1105,1426,1588],
     [3081,2552,2656,2267,2501,1688,1210,1137,1490,1345,1546,1719]],
    # F3 (Hz)
    [[3000,2684,2691,2605,2601,2522,2538,2459,2434,2343,2550,1710],
     [3372,3053,3047,2979,2972,2815,2824,2828,2827,2735,2933,1929],
     [3702,3403,3323,3310,3289,2950,2982,2987,3072,2988,3145,2143]],
    # F4 (Hz)
    [[3657,3618,3649,3677,3624,3687,3486,3384,3400,3357,3557,3334],
     [4352,4334,4319,4294,4290,4299,3923,3927,4052,4115,4092,3914],
     [4572,4575,4422,4671,4409,4307,3919,4167,4328,4276,4320,3788]],
    ], dtype=float)

# shape: (6 measures, 3 groups, 12 vowels)
# index as data[measures.index('F1'), groups.index('W'), vowels.index('ae')]

# ---- Define Functions that we need to compute the final spectrum (bandwidths, transfer functions, glottal source) ----

# use bandwidth estimation from Hawks and Miller 1995; bandwidths determine how narrow spectral peaks are
def get_bandwidth(formant, f0 = 175):
    S = 1 + 0.25 * (f0-132)/88 # formant bandwidths around 25% larger in women; interpolate linearly between avg f0 for men and f0 for women

    # bandwidths determined by piecewise function, formants below 500 and above 500 Hz
    if formant < 500: 
        k = 165.327516
        xs = np.array([-6.7363e-1, 1.8087e-3, -4.5220e-6, 7.4951e-9, -4.7022e-12])
    else: 
        k = 15.8146139
        xs = np.array([8.1016e-2, -9.7973e-5, 5.2873e-8, -1.0710e-11, 7.9153e-16])

    return S * (k + np.sum([xs[n]*formant**(n+1) for n in np.arange(0, len(xs))]))


# transfer function computed from Cascade model of vocal tract resonance as shown in Klatt 1989
# Tc = 1e-4 # inverse gives sampling rate
def T(f, pole, BW = 50, Tc = 1e-4): # transfer function in frequency domain
    ''''''
    B= 2*np.exp(-np.pi*BW*Tc)*np.cos(2*np.pi*pole*Tc)
    C = -np.exp(-2*np.pi*BW*Tc)
    A = 1 - B - C

    z = np.exp(2j * np.pi * f * Tc)
    # transfer function (in frq)
    return A / (1 - B/z - C *z**-2) # returns single transfer function across range of f-values, with a given pole

def compute_H(f_range, poles, bandwidths): # computes total transfer function across a range of frequencies (sums each resonant peak)
    temp_H = np.ones_like(f_range, dtype = 'complex')
    for i, (p, b) in enumerate(zip(poles, bandwidths)):
        temp_H *= T(f_range, p, b)
    return temp_H

# Walker, Murphy 2003 'Analytical Spectrum Formulation of Glottal Flow'; computes analytic glottal spectrum by analytic Fourier transform of Rosenburg glottal pulse waveform
def analytic_glottal_spectrum(f, oq = 0.8, f0 = None): # default in paper is t1 = 4e-3, t2 = 5e-3 (which corresponds to open-quotient (oq) of 0.8); default f0 = 100 Hz in paper
    # T2 = 1/f0
    # T1 = T2/1.25
    if f0 != None: # computes glottal pulse based on provided f0 (and thus provided Period), instead of default params
        T2  = 1/f0
        T1 = T2*oq
    else: # f0 = 100 Hz (implicitly) bc f0 = 1/T2
        T1 = 4e-3
        T2 = 5e-3

    delT = T2-T1
    phase = np.pi * T1 / (2 * delT)

    f1, f2 = 1/(2 *T1), 1/(4*delT)
    G1 = ( # note: np.sinc(x) automatically computes sinc(pi*x) / pi * x, so pi not included below
        0.5 * T1 * np.exp(-T1*f*np.pi*1j) * np.sinc(f*T1)
        - 0.25 * T1 * (np.sinc(T1*(f - f1))*np.exp(-1j*np.pi*(f-f1)*T1)
                    +  np.sinc(T1*(f + f1))*np.exp(-1j*np.pi*(f+f1)*T1))
    )
    G2 = delT / 2 * (np.sinc(delT * (f-f2))*np.exp(-1j*phase)*np.exp(-1j*np.pi*(f-f2)*(T1+T2))
                     + np.sinc(delT*(f+f2))*np.exp(1j*phase)*np.exp(-1j*np.pi*(f+f2)*(T1+T2))
                     )
    return G1 + G2

# we wish to plot the appropriate transfer function for men, women, and children
def plot_transfer_function(vowel = 'a', group = 'Woman', f0 = 215, max_f = 4500, oq = 0.8, Tc = 1e-4,
                           plot_time = False, plot_formants = True, plot_envelope = True, plot_harmonics = True, plot_only_transfer = False):
    '''---parameters---
    vowel: str; desired vowel sound
    f0: float; fundamental frequency of glottal vibration
    group: str; Man ('M'), Woman ('W'), Child ('C')
    max_f: float; maximum frequency to be plotted
    Tc: float; inverse gives sampling rate
    '''
    # Nyquist frequency gives upper limit of 5000 Hz for sampling rate of 1 / 1e-4
    Nyquist_f = (1/Tc) / 2
    if max_f > Nyquist_f:
        max_f = 5e3 # fixes upper bound, as aliasing makes solutions begin to repeat above 5 kHz (bc sampling rate is 1e-4)

    v_i, g_i = vowel_labels.index(vowel), groups.index(group)

    # if choose_f0 == False: 
    #     f0 = data[1, g_i, v_i]
    print('Hillenbrand Measured F0:', data[1, g_i, v_i])
    print('Chosen f0:', f0)

    formants = np.array(data[2:6, g_i, v_i])
    bandwidths = np.array([get_bandwidth(f, f0) for f in formants])
    # print('f0', f0)

    f_range = np.linspace(f0, int(max_f), 100000)

    H = np.ones_like(f_range, dtype = 'complex')

    # fig, (ax, ax2) = plt.subplots(ncols = 2) # if plotting time-domain representation as well
    fig, ax = plt.subplots()

    for (f, b) in zip(formants, bandwidths):
        H *= T(f_range, f, b)

    # print(H)
    H_dB = 20 * np.log10(np.abs(H)) # put transfer function in dB for plotting

    glottal_source = analytic_glottal_spectrum(f_range, f0 = f0, oq = oq)
    glottal_source /= np.max(np.abs(glottal_source)) # normalize source to start at 0 dB
    # glottal_source *= 1e3 # normalize glottal source to start at 60 dB; ideally more representative of what this spectrum might look like?
    glottal_dB = 20 * np.log10(np.abs(glottal_source)) # convert to dB for plotting
    # glottal_dB = -12 * np.log2(f_range/f0) # alternative (simpler) shape in f-domain, but yields not quite correct glottal pulse shape in time-domain representation

    print('Vocal Formants:', formants)
    # ax.plot(f_range, glottal_dB, lw = 1, ls = 'dotted', label = 'Glottal Source')
    if plot_only_transfer: 
        ax.plot(f_range, H_dB, lw = 1, label = 'Transfer Function T(f)', c = 'k')
        ax.set_title('Transfer Function T(f)')
        plot_envelope, plot_harmonics, plot_formants = False, False, False
    a = 1 if plot_envelope else 0
    ax.plot(f_range, H_dB + glottal_dB, lw = 1, label = 'Total: T(f)S(f)', c = 'k', alpha = a)

    f0_harmonics = f0*np.arange(1, max_f/f0, 1)
    
    eps = 1e-1
    
    # note: H*G must be plotted before calling get_ybound, otherwise ax just returns minimum of unplotted axis label (0), which leads to undesired plotting
    ymin, ymax = ax.get_ybound() 
    ymin = np.min(H_dB) if plot_only_transfer else ymin # if plotting only T(f), set minimum accordingly
    
    # with a low threshold, each check finds multiple y-values within the threshold, so we just choose the first one ([0]) for convenience
    
    if plot_harmonics: # plot harmonics of f0
        y = np.array([20 * np.log10(np.abs(H[np.abs(f_range - x) <= eps]) * np.abs(glottal_source[np.abs(f_range - x) <= eps]))[0] for x in f0_harmonics])
        markerlines, stemline, baseline = ax.stem(f0_harmonics, y, 
                markerfmt = 'None', bottom = ymin, label = 'Harmonics of f0')
        markerlines.set_alpha(0)
        stemline.set_alpha(1)
        stemline.set_linewidth(1)
        baseline.set_alpha(0)

    if plot_formants: # plot vertical lines to show formants of vocal tract
        y2 = [20 * np.log10(np.abs(H[np.abs(f_range - x) <= eps])* np.abs(glottal_source[np.abs(f_range - x) <= eps]))[0] for x in formants]
        mlines, sline, bline = ax.stem(formants, y2, 
                markerfmt = 'None', bottom = ymin, label = 'Vowel Formants')
        
        # vertical formant labels computed after tight_layout to avoid override ; text appears at location x = F, y = ymin + 5 for vertical offset (dB)
        for i, F in enumerate(formants): 
            ax.text(F, ymin + 5, f"F{i+1} = {F} Hz", 
                    rotation=90, verticalalignment='bottom', horizontalalignment='right', fontsize=8) # 'baseline' - bottom of text at bottom of y-axis
    
        mlines.set_alpha(0)
        sline.set_alpha(1)
        sline.set_linewidth(1.25)
        sline.set_color('Red')
        bline.set_alpha(0)
    # # print(x)
    # ax.vlines(x, ymin = -60, ymax = 20 * np.log10(np.abs(H[f_range == x])) -12 * np.log2(x/f0), lw = 1)

    # ax.set_title(f'{group} Saying {vowels[v_i]} as in {vowel_labels[v_i]}; f0 = {f0} Hz')
    ax.set_xlim(0, max_f)
    ax.set_ylim(ymin, None)
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('Relative Amplitude (dB)')

    ax.tick_params(axis = 'y')
    ax.yaxis.set_major_locator(ticker.MultipleLocator(10))
    ax.xaxis.set_minor_locator(ticker.MultipleLocator(100))

    ax.grid(True, alpha = 0.5)
    ax.set_title('P(f): Sound Pressure Spectrum')
    
    # # ax.legend()

    # compute time-domain representation using the IRFFT of pressure spectrum 
    fs = 16e3 # sample rate
    n_periods = 4 # choose how many periods to plot
    N = int(round(fs / f0)) * n_periods    # choose N so harmonics land on exact bins

    bin_width = fs / N
    freqs = np.fft.rfftfreq(N, d=1/fs)     # bin frequencies, length N//2+1
    spectrum = np.zeros(len(freqs), dtype=complex)

    eps = 1e-1
    n_harmonics = int(fs/2 / f0)
    for k in range(1, n_harmonics + 1):
        f_k = k * f0
        bin_idx = int(round(f_k / bin_width))
        if bin_idx >= len(freqs): # make sure idx is within our frequency range
            continue
        if f_k < max_f: # make sure sampled harmonic is within frequency range
            spectrum[bin_idx] = H[np.abs(f_range - f_k) <= eps][0] * glottal_source[np.abs(f_range - f_k) <= eps][0]
        else: 
            spectrum[bin_idx] = 0

    # ---- commented out: makes sure that sampled spectrum in each bin corresponds to the expected analytic net transfer function: H(f)G(f) -----
    # mask = np.nonzero(np.abs(spectrum))
    # ax.plot(freqs[mask], 20 * np.log10(np.abs(spectrum[mask]))) # it does, so we don't need to display this in the final figure

    signal = np.fft.irfft(spectrum, n=N)  # real time-domain output

    t = np.arange(N) / fs* 1e3 # ms; sample N times at periods determined by inverse of sampling frequency fs, then convert to ms

    if plot_time:
        fig2, ax2 = plt.subplots()
        ax2.plot(t, signal, lw = 1, color = 'k')
        ax2.set_title('Time-Domain Signal')
        ax2.set_xlabel('Time (ms)')
        ax2.set_ylim(np.min(signal)*1.5, np.max(signal)*1.5)

        fig2.tight_layout()

    fig.tight_layout()
    

    # define functions necessary to play audio
    duration = 0.6
    reps = int(np.ceil(duration*fs / N))
    sig = np.tile(signal, reps)[:int(duration*fs)]
    sig = sig / (np.max(np.abs(sig)) + 1e-9)   # normalize signal 

    # short fade in/out to avoid audible clicks at start/end
    fade = int(0.01 * fs)
    env = np.ones_like(sig)
    env[:fade] = np.linspace(0, 1, fade)
    env[-fade:] = np.linspace(1, 0, fade)
    audio_sig = sig * env

    # --- Cell: widget with play button ---
    play_button = Button(description='▶ Play sound', button_style='success')
    audio_out = Output()

    def on_play_clicked(b):
        audio_out.clear_output(wait=True)
        with audio_out:
            display(Audio(audio_sig, rate=fs, autoplay=True))

    play_button.on_click(on_play_clicked)

    display(VBox([play_button, audio_out]))

# make the plot interactive; can choose the vowel, person speaking, fundamental (f0)
interact(
    plot_transfer_function,
    vowel = Dropdown(options = vowel_labels, value = 'hod, /α/', description = 'Vowel: '),
    group = Dropdown(options = groups, value = 'Woman', description = 'Person Speaking:'), 
    f0 = IntSlider(min = 70, max = 450, step = 1, value = data[1, 1, vowel_labels.index('hod, /α/')]),
    oq = fixed(0.8), # open quotient fixed to equal that found in Walker Murphy paper
    Tc = fixed(1e-4), # inverse = sampling rate
    max_f = fixed(4500), # plots fixed to have max of 4500; note that max_f cannot go above Nyquist freq bc of aliasing
)


interactive(children=(Dropdown(description='Vowel: ', index=5, options=('heed, /i/', 'hid, /I/', 'hayed, /e/',…

<function __main__.plot_transfer_function(vowel='a', group='Woman', f0=215, max_f=4500, oq=0.8, Tc=0.0001, plot_time=False, plot_formants=True, plot_envelope=True, plot_harmonics=True, plot_only_transfer=False)>